In [ ]:
# --- GLOBAL VARIABLES ---
PROJECT_ID = "power-gridintelligence-bq"  # Change this to your project ID
DATASET_ID = "grid_reliability_gold"      # Your dataset name
LOCATION = "us-east4"                    # e.g., us-central1 or us-east4
PROJECT_NUMBER = "630496998101"
from google.cloud import bigquery
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

client = bigquery.Client(project=PROJECT_ID)

# Create Dataset if it doesn't exist
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f"Dataset {dataset_id} is ready.")

Dataset power-gridintelligence-bq.grid_reliability_gold is ready.


In [ ]:
def generate_prd_telemetry(num_rows=20):
    start_time = datetime.now().replace(hour=14, minute=0, second=0, microsecond=0)
    data = []

    for i in range(num_rows):
        ts = start_time + timedelta(minutes=i * 5)
        asset_id = "COMP-TX-VALLEY-01"

        # Default Normal Values
        psi, temp, vib = 855, 112, 0.02
        is_failure = 0

        # Inject the PRD Failure Case at index 3 (14:10:00)
        if i == 2:
            psi, temp, vib = 645, 158, 0.85
            is_failure = 1

        data.append({
            "timestamp": ts.strftime("%I:%M:%S %p"),
            "asset_id": asset_id,
            "psi": psi,
            "temp_f": temp,
            "vibration": vib,
            "is_failure": is_failure
        })
    return pd.DataFrame(data)

df = generate_prd_telemetry()
# ... (load to BigQuery as shown in previous steps)

In [ ]:
%%bigquery --project $PROJECT_ID
CREATE OR REPLACE MODEL `grid_reliability_gold.stator_failure_classifier`
OPTIONS(
  MODEL_TYPE = 'BOOSTED_TREE_CLASSIFIER',
  INPUT_LABEL_COLS = ['is_failure'],
  BOOSTER_TYPE = 'GBTREE',
  NUM_PARALLEL_TREE = 3,
  MAX_ITERATIONS = 20,
  MODEL_REGISTRY = 'VERTEX_AI',
  VERTEX_AI_MODEL_ID = 'stator_failure_classifier_v1',
  VERTEX_AI_MODEL_VERSION_ALIASES = ['production', 'tallgrass_poc']
) AS
SELECT
  psi, temp_f, vibration, is_failure
FROM
  `grid_reliability_gold.telemetry_raw`;

Query is running:   0%|          |

""


In [ ]:
%%bigquery --project $PROJECT_ID
SELECT
  timestamp AS TIMESTAMP,
  asset_id AS ASSET_ID,
  psi AS PSI,
  temp_f AS TEMP_F,  -- Changed from TEMP(°F)
  CONCAT(CAST(vibration AS STRING), 'mm') AS VIBRATION,
  CASE
    WHEN predicted_is_failure = 1 THEN 'Critical Alert'
    ELSE 'Normal'
  END AS STATUS,
  CASE
    WHEN predicted_is_failure = 1 THEN ''
    ELSE '—'
  END AS ACTION
FROM
  ML.PREDICT(
    MODEL `grid_reliability_gold.stator_failure_classifier`,
    (SELECT * FROM `grid_reliability_gold.telemetry_raw`)
  )
ORDER BY timestamp DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,TIMESTAMP,ASSET_ID,PSI,TEMP_F,VIBRATION,STATUS,ACTION
0,12:55:00 PM,COMP-TX-VALLEY-01,846,115,0.04mm,Normal,—
1,12:50:00 PM,COMP-TX-VALLEY-01,846,112,0.02mm,Normal,—
2,12:45:00 PM,COMP-TX-VALLEY-02,846,116,0.02mm,Normal,—
3,12:40:00 PM,COMP-TX-VALLEY-01,849,108,0.04mm,Normal,—
4,12:35:00 PM,COMP-TX-VALLEY-01,859,109,0.04mm,Normal,—
5,12:30:00 PM,COMP-TX-VALLEY-02,856,113,0.03mm,Normal,—
6,12:25:00 PM,COMP-TX-VALLEY-01,848,115,0.04mm,Normal,—
7,12:20:00 PM,COMP-TX-VALLEY-01,849,109,0.04mm,Normal,—
8,12:15:00 PM,COMP-TX-VALLEY-02,853,108,0.03mm,Normal,—
9,12:10:00 PM,COMP-TX-VALLEY-01,848,111,0.04mm,Normal,—
